# NeoOLAF native EventStoryLine layer ablation — one document v1.7

This v1.7 freezes the high-recall atomic event inventory and exhaustive pair pool, then focuses entirely on relation quality: text-forward PLOT_LINK orientation, one global plot-role pass, compact six-target source batches, slightly relaxed narrative-link criteria, and targeted PRECONDITION-bias calibration. Reverse textual directions are never classified. No file under `src/neoolaf` is modified.


In [1]:
from __future__ import annotations

import json
import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


def first_existing_path(env_name: str, candidates: list[Path]) -> Path:
    raw = os.environ.get(env_name, "").strip().strip('"').strip("'")
    if raw:
        path = Path(raw).expanduser().resolve()
        if path.is_file():
            return path
        raise FileNotFoundError(f"{env_name} points to a missing file: {path}")
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {env_name}. Tried:\n"
        + "\n".join(str(Path(x).expanduser().resolve()) for x in candidates)
    )


def first_existing_dir(env_name: str, candidates: list[Path]) -> Path:
    raw = os.environ.get(env_name, "").strip().strip('"').strip("'")
    if raw:
        path = Path(raw).expanduser().resolve()
        if path.is_dir():
            return path
        raise FileNotFoundError(f"{env_name} points to a missing directory: {path}")
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find {env_name}. Tried:\n"
        + "\n".join(str(Path(x).expanduser().resolve()) for x in candidates)
    )


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def strip_gold(record: dict) -> dict:
    return {k: v for k, v in record.items() if k not in {"entities", "relations"}}


def prepare_eventstoryline_views(
    dataset_jsonl: Path,
    selected_document_id: str,
    input_jsonl: Path,
    gold_jsonl: Path,
    smoke5_input_jsonl: Path,
    smoke5_gold_jsonl: Path,
) -> None:
    selected = None
    smoke5 = []
    with dataset_jsonl.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if len(smoke5) < 5:
                smoke5.append(record)
            if record.get("document_id") == selected_document_id:
                selected = record
            if selected is not None and len(smoke5) >= 5:
                break

    if selected is None:
        raise KeyError(
            f"Document {selected_document_id!r} was not found in {dataset_jsonl}"
        )
    if len(smoke5) != 5:
        raise RuntimeError(f"Expected at least 5 EventStoryLine records, found {len(smoke5)}")

    write_jsonl(input_jsonl, [strip_gold(selected)])
    write_jsonl(gold_jsonl, [selected])
    write_jsonl(smoke5_input_jsonl, [strip_gold(row) for row in smoke5])
    write_jsonl(smoke5_gold_jsonl, smoke5)


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# Portable bundle preflight: v1.7 depends on v1.6 -> v1.5 -> v1.3 -> DocRED helper adapters.
# These files are experiment adapters only; NeoOLAF src itself remains untouched.
REQUIRED_LOCAL_HELPERS = [
    "eventstoryline_native_ablation_v1_7.py",
    "eventstoryline_native_ablation_v1_6.py",
    "eventstoryline_native_ablation_v1_5.py",
    "eventstoryline_native_ablation_v1_3.py",
    "docred_native_ablation.py",
    "docred_native_ablation_v3.py",
    "docred_native_ablation_v4.py",
]
missing_local_helpers = [name for name in REQUIRED_LOCAL_HELPERS if not (TOOLS_DIR / name).is_file()]
if missing_local_helpers:
    raise FileNotFoundError(
        "Incomplete portable EventStoryLine bundle. Missing helper files under "
        f"{TOOLS_DIR}:\n" + "\n".join(missing_local_helpers)
    )

from eventstoryline_native_ablation_v1_7 import (
    RELATION_IDS,
    analyze_run,
    gold_event_index,
    indexed_token_table,
    load_layer_states,
    project_event_label,
    read_json,
    read_jsonl,
    run_native_pipeline,
    seed_ontology_summary,
    resolve_source_centric_cache_dir,
)

print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = C:\Users\galencarmedeiro\NeoOLAF


c:\Users\galencarmedeiro\NeoOLAF\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\galencarmedeiro\NeoOLAF\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

In [2]:
# New shared RAGTree preprocessed dataset directory.
# The environment variable remains the safest override if the repository moves again.
RAGTREE_PREPROCESSED_DIR = first_existing_dir(
    "RAGTREE_PREPROCESSED_DIR",
    [
        Path(r"C:\Users\galencarmedeiro\\RAGTree\preprocessed"),
        PROJECT_ROOT.parent / "RAGTree/preprocessed",
        PROJECT_ROOT.parent / "ragtree/preprocessed",
        PROJECT_ROOT / "../RAGTree/preprocessed",
        PROJECT_ROOT / "../ragtree/preprocessed",
    ],
)

DATASET_PATHS = {
    "CausalBank": RAGTREE_PREPROCESSED_DIR / "causalbank.jsonl",
    "DocRED": RAGTREE_PREPROCESSED_DIR / "docred_causal.jsonl",
    "EventStoryLine": RAGTREE_PREPROCESSED_DIR / "eventstoryline.jsonl",
    "FinCausal": RAGTREE_PREPROCESSED_DIR / "fincausal.jsonl",
    "MAVEN-ERE": RAGTREE_PREPROCESSED_DIR / "maven_ere.jsonl",
}
EVENTSTORYLINE_DATASET_JSONL = DATASET_PATHS["EventStoryLine"]
SELECTED_DOCUMENT_ID = "EventStoryLine - 1_10ecbplus"

# Same seed ontology used by the RAGTree EventStoryLine experiments.
# Override explicitly with EVENTSTORYLINE_ONTOLOGY_PATH when needed.
ONTOLOGY_PATH = first_existing_path(
    "EVENTSTORYLINE_ONTOLOGY_PATH",
    [
        PROJECT_ROOT.parent / "RAGTree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT.parent / "ragtree/data/ontology/OWLTime/time.ttl",
        Path(r"C:\Users\galencarmedeiro\Documents\git\postdoc\RAGTree\data\ontology\OWLTime\time.ttl"),
        PROJECT_ROOT / "../RAGTree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT / "../ragtree/data/ontology/OWLTime/time.ttl",
    ],
)

# Controlled normalized benchmark relation schema. This is not the seed ontology.
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/eventstoryline_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/eventstoryline_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_profile_native_ablation_v1_7.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_eventstoryline_native_ablation_v1_7.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_task_guidance_v1_7.json"

RUNS_ROOT = NOTEBOOK_DIR / "runs/eventstoryline_native_layer_ablation"
RUN_DIR = RUNS_ROOT / "document_1_10ecbplus_v1_7_owltime_forward_plot_calibrated"
PREPARED_DIR = RUNS_ROOT / "_prepared_eventstoryline_v1_7"
INPUT_JSONL = PREPARED_DIR / "eventstoryline_one_input_v1_7.jsonl"
GOLD_JSONL = PREPARED_DIR / "eventstoryline_one_gold_v1_7.jsonl"
SMOKE5_INPUT_JSONL = PREPARED_DIR / "eventstoryline_smoke5_input_v1_7.jsonl"
SMOKE5_GOLD_JSONL = PREPARED_DIR / "eventstoryline_smoke5_gold_v1_7.jsonl"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Fixed-source relation requests are independent and run concurrently in Layer 2.
WORKERS = 16
REASONING_EFFORT = "minimal"
RUN_PIPELINE = True
CLEAN_RUN_DIR = True

# v1.7: short, portable Layer-2 cache (same non-critical cache fix). Override only when desired.
LAYER2_CACHE_DIR = resolve_source_centric_cache_dir()

print("RAGTree preprocessed root:", RAGTREE_PREPROCESSED_DIR)
for dataset_name, dataset_path in DATASET_PATHS.items():
    print(f"{dataset_name:14s} -> {dataset_path}")
print("Selected EventStoryLine document:", SELECTED_DOCUMENT_ID)
print("OWL-Time seed ontology:", ONTOLOGY_PATH)
print("Run dir:", RUN_DIR)
print("Model:", MODEL_NAME)
print("API key available:", bool(API_KEY))
print("Layer-2 short cache:", LAYER2_CACHE_DIR)
print("Layer-2 cache path chars:", len(str(LAYER2_CACHE_DIR)))


RAGTree preprocessed root: C:\Users\galencarmedeiro\RAGTree\preprocessed
CausalBank     -> C:\Users\galencarmedeiro\RAGTree\preprocessed\causalbank.jsonl
DocRED         -> C:\Users\galencarmedeiro\RAGTree\preprocessed\docred_causal.jsonl
EventStoryLine -> C:\Users\galencarmedeiro\RAGTree\preprocessed\eventstoryline.jsonl
FinCausal      -> C:\Users\galencarmedeiro\RAGTree\preprocessed\fincausal.jsonl
MAVEN-ERE      -> C:\Users\galencarmedeiro\RAGTree\preprocessed\maven_ere.jsonl
Selected EventStoryLine document: EventStoryLine - 1_10ecbplus
OWL-Time seed ontology: C:\Users\galencarmedeiro\RAGTree\data\ontology\OWLTime\time.ttl
Run dir: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\eventstoryline_native_layer_ablation\document_1_10ecbplus_v1_7_owltime_forward_plot_calibrated
Model: openai/gpt-oss-20b
API key available: True
Layer-2 short cache: C:\Users\GALENC~1\AppData\Local\Temp\neoolaf_esl16
Layer-2 cache path chars: 50


## 2. Preflight: stream the real EventStoryLine JSONL, strip gold, and validate the relation-focused configuration


In [3]:

# v1.7 Windows-path/cache preflight. Cache is optional for correctness, but
# verify the chosen short directory when the filesystem allows it.
cache_probe_ok = False
cache_probe_error = None
try:
    LAYER2_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    probe = LAYER2_CACHE_DIR / "probe.json"
    probe.write_text('{"ok": true}', encoding="utf-8")
    probe.unlink(missing_ok=True)
    cache_probe_ok = True
except OSError as exc:
    cache_probe_error = f"{type(exc).__name__}: {exc}"
print("Layer-2 cache writable:", cache_probe_ok)
if cache_probe_error:
    print("Layer-2 cache warning (non-fatal in v1.7):", cache_probe_error)
assert len((LAYER2_CACHE_DIR / ("a" * 24 + ".json")).__str__()) < 180 or os.name != "nt", (
    "Layer-2 cache path is unexpectedly long on Windows; set "
    "NEOOLAF_EVENTSTORYLINE_CACHE_DIR to a shorter directory."
)

missing_datasets = [str(path) for path in DATASET_PATHS.values() if not path.is_file()]
if missing_datasets:
    raise FileNotFoundError(
        "Missing one or more preprocessed datasets under "
        f"{RAGTREE_PREPROCESSED_DIR}:\n" + "\n".join(missing_datasets)
    )

# Stream the real EventStoryLine JSONL and materialize only the selected document + first 5.
# Gold fields are stripped before the pipeline input is written.
prepare_eventstoryline_views(
    dataset_jsonl=EVENTSTORYLINE_DATASET_JSONL,
    selected_document_id=SELECTED_DOCUMENT_ID,
    input_jsonl=INPUT_JSONL,
    gold_jsonl=GOLD_JSONL,
    smoke5_input_jsonl=SMOKE5_INPUT_JSONL,
    smoke5_gold_jsonl=SMOKE5_GOLD_JSONL,
)

required = [
    INPUT_JSONL, GOLD_JSONL, SMOKE5_INPUT_JSONL, SMOKE5_GOLD_JSONL,
    ONTOLOGY_PATH, RELATION_CATALOG, RELATION_ALIASES,
    PROFILE_PATH, GUIDANCE_PATH, TASK_GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

input_rows = read_jsonl(INPUT_JSONL)
gold_rows = read_jsonl(GOLD_JSONL)
smoke_input_rows = read_jsonl(SMOKE5_INPUT_JSONL)
smoke_gold_rows = read_jsonl(SMOKE5_GOLD_JSONL)
assert len(input_rows) == 1 and len(gold_rows) == 1
assert len(smoke_input_rows) == 5 and len(smoke_gold_rows) == 5
assert "entities" not in input_rows[0] and "relations" not in input_rows[0]
assert all("entities" not in row and "relations" not in row for row in smoke_input_rows)
assert input_rows[0]["document_id"] == gold_rows[0]["document_id"]
assert [x["document_id"] for x in smoke_input_rows] == [x["document_id"] for x in smoke_gold_rows]

profile = read_json(PROFILE_PATH)
task = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)
gold = gold_rows[0]
seed_summary = seed_ontology_summary(ONTOLOGY_PATH)
l1_cfg = profile["layers"]["layer01_linguistic_expression_extraction"]
l2_cfg = profile["layers"]["layer02_candidate_enrichment"]

print("Full EventStoryLine dataset:", EVENTSTORYLINE_DATASET_JSONL)
print("Document:", input_rows[0]["document_id"], "-", input_rows[0]["title"])
print("Source characters:", len(input_rows[0]["text"]))
print("Sentences:", len(input_rows[0]["sentences"]))
print("Source tokens:", sum(len(x) for x in input_rows[0]["tokens"]))
print("Gold events (not exposed to pipeline):", len(gold["entities"]))
print("Gold evaluated relations:", sum(len(v) for k, v in gold["relations"].items() if k in RELATION_IDS))
print("Ignored null pairs:", len(gold["relations"].get("null", [])))
print("OWL-Time classes loaded:", seed_summary["class_count"])
print("OWL-Time properties loaded:", seed_summary["property_count"])
print("Controlled task relations:", catalog["property_count"])
print("Relation IDs:", task["allowed_relation_ids"])
print("Layer 1 sentence workers:", l1_cfg["sentence_workers"])
print("Layer 1 coverage reviews:", l1_cfg["coverage_review_passes"])
print("Pair strategy:", l1_cfg["pair_strategy"])
print("Exhaustive-event threshold:", l1_cfg["exhaustive_event_threshold"])
print("Relation-generation LLM calls:", l1_cfg["relation_generation_llm_calls"])
print("Positive evidence required:", l2_cfg["evidence_required_for_positive"])
print("NONE filtered before Layer 3:", l2_cfg["filter_none_before_layer03"])
print("Global-trigger projection fallback:", profile["benchmark_projection"]["fallback_unique_trigger"])
print("Mention-free relation schemas injected:", len(profile["relations"]["allowed"]))
print("Five-document input IDs:", [row["document_id"] for row in smoke_input_rows])

assert seed_summary["class_count"] > 0 or seed_summary["property_count"] > 0
assert catalog["property_count"] == 2
assert set(task["allowed_relation_ids"]) == set(RELATION_IDS)
assert profile["relations"]["allowed"] == []
assert l1_cfg["relation_generation_llm_calls"] == 0
assert l1_cfg["deterministic_unordered_pair_pool"] is True
assert l2_cfg["filter_none_before_layer03"] is True
assert l2_cfg["evidence_required_for_positive"] is True
assert profile["benchmark_projection"]["fallback_unique_trigger"] is False
assert profile["anti_cheating"]["direct_eventstoryline_extraction"] is False
assert profile["anti_cheating"]["source_event_anchoring"] is False
assert profile["anti_cheating"]["gold_pair_hints"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False
assert profile["benchmark_projection"]["same_seed_ontology_as_ragtree"] is True

# Prompt examples are synthetic and schema-exact; no dataset gold pair is embedded.
assert len(task["source_centric_schema_examples"]) >= 5
assert {x["relation"] for x in task["source_centric_schema_examples"]} >= {
    "PRECONDITION", "FALLING_ACTION", "NONE"
}

display(Markdown("### Indexed source table used by Layer 1"))
print(indexed_token_table(input_rows[0]["sentences"], input_rows[0]["tokens"]))

assert profile["anti_cheating"]["gold_event_lexicon"] is False
assert profile["anti_cheating"]["gold_event_count"] is False
assert profile["anti_cheating"]["gold_relation_count"] is False

print("Atomic span review enabled:", l1_cfg.get("atomic_review_enabled"))
print("Pair-local direct-link mode:", l2_cfg.get("direct_link_only"))
print("Candidate closure enabled:", l2_cfg.get("closure_enabled"))
print("Broad NONE review:", l2_cfg.get("none_review_enabled"))
print("Positive verifier:", l2_cfg.get("positive_verifier_enabled"))
print("Fixed source per request:", l2_cfg.get("fixed_source_per_request"))
print("Target decisions:", l2_cfg.get("target_decisions"))


assert l2_cfg["strategy"] == "eventstoryline_forward_textual_plot_role_calibrated_source_batches"
assert l2_cfg["target_decisions"] == ["PRECONDITION", "FALLING_ACTION", "NONE"]
assert l2_cfg["none_review_enabled"] is False
assert l2_cfg["forward_textual_direction_only"] is True
assert l2_cfg["target_batch_size"] == 6
assert l2_cfg["plot_role_analysis_enabled"] is True
assert l2_cfg["positive_verifier_enabled"] is False
assert l2_cfg["closure_enabled"] is False
print("Layer 2 strategy:", l2_cfg["strategy"])
print("Source workers:", l2_cfg["source_workers"])
print("Missing-target retry:", l2_cfg["missing_target_retry"])
print("Forward-only direction:", l2_cfg["forward_textual_direction_only"])
print("Target batch size:", l2_cfg["target_batch_size"])
print("Global plot-role analysis:", l2_cfg["plot_role_analysis_enabled"])
print("Relax narrative links:", l2_cfg["relax_narrative_links"])
print("Label calibration:", l2_cfg["label_calibration_enabled"])
print("Broad NONE review:", l2_cfg["none_review_enabled"])
print("Positive verifier:", l2_cfg["positive_verifier_enabled"])


Layer-2 cache writable: True
Full EventStoryLine dataset: C:\Users\galencarmedeiro\RAGTree\preprocessed\eventstoryline.jsonl
Document: EventStoryLine - 1_10ecbplus - 1_10ecbplus
Source characters: 745
Sentences: 6
Source tokens: 147
Gold events (not exposed to pipeline): 15
Gold evaluated relations: 20
Ignored null pairs: 1
OWL-Time classes loaded: 23
OWL-Time properties loaded: 58
Controlled task relations: 2
Relation IDs: ['PRECONDITION', 'FALLING_ACTION']
Layer 1 sentence workers: 8
Layer 1 coverage reviews: 3
Pair strategy: adaptive
Exhaustive-event threshold: 25
Relation-generation LLM calls: 0
Positive evidence required: True
NONE filtered before Layer 3: True
Global-trigger projection fallback: False
Mention-free relation schemas injected: 0
Five-document input IDs: ['EventStoryLine - 1_10ecbplus', 'EventStoryLine - 1_11ecbplus', 'EventStoryLine - 1_12ecbplus', 'EventStoryLine - 1_13ecbplus', 'EventStoryLine - 1_14ecbplus']


### Indexed source table used by Layer 1

[S0] http : / / articles . latimes . com / 2013 / may / 03 / local / la - me - 0504 - lohan - rehab - 20130504
TOKENS 0=http 1=: 2=/ 3=/ 4=articles 5=. 6=latimes 7=. 8=com 9=/ 10=2013 11=/ 12=may 13=/ 14=03 15=/ 16=local 17=/ 18=la 19=- 20=me 21=- 22=0504 23=- 24=lohan 25=- 26=rehab 27=- 28=20130504

[S1] Lindsay Lohan checks into Betty Ford Center
TOKENS 0=Lindsay 1=Lohan 2=checks 3=into 4=Betty 5=Ford 6=Center

[S2] May 03 , 2013
TOKENS 0=May 1=03 2=, 3=2013

[S3] After skipping out on entering a Newport Beach rehabilitation facility and facing the prospect of arrest for violating her probation , Lindsay Lohan has checked into the Betty Ford Center to begin a 90 - day court - mandated stay in her reckless driving conviction .
TOKENS 0=After 1=skipping 2=out 3=on 4=entering 5=a 6=Newport 7=Beach 8=rehabilitation 9=facility 10=and 11=facing 12=the 13=prospect 14=of 15=arrest 16=for 17=violating 18=her 19=probation 20=, 21=Lindsay 22=Lohan 23=has 24=checked 25=into 26=the 27=Betty 28=Fo

## 3. Run the full native Layer 0--12 pipeline

In [4]:
if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    final_state = run_native_pipeline(
        project_root=PROJECT_ROOT,
        input_jsonl=INPUT_JSONL,
        ontology_path=ONTOLOGY_PATH,
        profile_path=PROFILE_PATH,
        guidance_path=GUIDANCE_PATH,
        task_guidance_path=TASK_GUIDANCE_PATH,
        relation_catalog_path=RELATION_CATALOG,
        relation_aliases_path=RELATION_ALIASES,
        run_dir=RUN_DIR,
        model_name=MODEL_NAME,
        api_key=API_KEY,
        host=OPENROUTER_HOST,
        workers=WORKERS,
        reasoning_effort=REASONING_EFFORT,
        verbose=True,
        clean_run_dir=CLEAN_RUN_DIR,
    )
    print("Full native run completed.")
    print("Layer-2 cache used:", resolve_source_centric_cache_dir())
else:
    print("RUN_PIPELINE=False: reusing", RUN_DIR)

[NeoOLAF] Run directory: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\eventstoryline_native_layer_ablation\document_1_10ecbplus_v1_7_owltime_forward_plot_calibrated
[NeoOLAF] from_layer=0, to_layer=None, skip_layers=None
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer00_preprocessing', 'layer01_linguistic_expression_extraction', 'layer02_candidate_enrichment', 'layer03_candidate_typing_resolution', 'layer04_candidate_relation_extraction', 'layer05_candidate_triple_generation', 'layer06_concept_relation_induction', 'layer07_hierarchisation', 'layer08_axiom_schemata_extraction', 'layer09_general_axiom_extraction', 'layer10_validation_reasoning', 'layer11_inference_completion', 'layer12_serialization']
[NeoOLAF] Layer 0/12: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.00s
[NeoOLAF] Layer 1/12: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_lingu

[NeoOLAF] Finished layer: layer03_candidate_typing_resolution in 0.13s
[NeoOLAF] Layer 4/12: layer04_candidate_relation_extraction

[NeoOLAF] Starting layer: layer04_candidate_relation_extraction
[NeoOLAF][Layer 4] strategy=structured_exact_then_native_parallel_fallback; parallel_workers=8; attempts=1
[NeoOLAF] Finished layer: layer04_candidate_relation_extraction in 0.08s
[NeoOLAF] Layer 5/12: layer05_candidate_triple_generation

[NeoOLAF] Starting layer: layer05_candidate_triple_generation


[NeoOLAF] Finished layer: layer05_candidate_triple_generation in 0.01s
[NeoOLAF] Layer 6/12: layer06_concept_relation_induction

[NeoOLAF] Starting layer: layer06_concept_relation_induction
[NeoOLAF][Layer 6] deterministic ontology-aware concept induction for 20 node candidates; no LLM calls.
[NeoOLAF][Layer 6] deterministic ontology-aware relation induction for 109 relation candidates; no LLM calls.
[NeoOLAF] Finished layer: layer06_concept_relation_induction in 0.01s
[NeoOLAF] Layer 7/12: layer07_hierarchisation

[NeoOLAF] Starting layer: layer07_hierarchisation
[NeoOLAF] Finished layer: layer07_hierarchisation in 0.00s
[NeoOLAF] Layer 8/12: layer08_axiom_schemata_extraction

[NeoOLAF] Starting layer: layer08_axiom_schemata_extraction
[NeoOLAF][Layer 8] strategy=ontology_aware_axiom_schema_generation
[NeoOLAF] Finished layer: layer08_axiom_schemata_extraction in 0.00s
[NeoOLAF] Layer 9/12: layer09_general_axiom_extraction

[NeoOLAF] Starting layer: layer09_general_axiom_extraction
[N

[NeoOLAF] Finished layer: layer10_validation_reasoning in 0.01s
[NeoOLAF] Layer 11/12: layer11_inference_completion

[NeoOLAF] Starting layer: layer11_inference_completion
[NeoOLAF][Layer 11] strategy=ontology_aware_semantic_completion
[NeoOLAF][Layer 11] deterministic completion; max_concurrency=16; no LLM calls.
[NeoOLAF] Finished layer: layer11_inference_completion in 0.00s
[NeoOLAF] Layer 12/12: layer12_serialization

[NeoOLAF] Starting layer: layer12_serialization
[NeoOLAF] Exports written to: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\eventstoryline_native_layer_ablation\document_1_10ecbplus_v1_7_owltime_forward_plot_calibrated\exports
[NeoOLAF] Finished layer: layer12_serialization in 0.25s
[NeoOLAF] Pipeline finished in 291.45s
[NeoOLAF] Saved checkpoint: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\eventstoryline_native_layer_ablation\document_1_10ecbplus_v1_7_owltime_forward_plot_calibrated\checkpoints\after_selected_pipeline.pkl.gz
[NeoO

### Runtime evidence saved

- `run_logs/layer02_plot_roles.json` records the global SETUP/PIVOT/AFTERMATH analysis.
- `run_logs/layer02_forward_direction_filter.json` records the one-way textual orientation applied to every unordered pair.
- `run_logs/layer02_source_centric_decisions.json` records every classified forward pair.
- `run_logs/layer02_relation_calibration.json` records any PRECONDITION/FALLING_ACTION calibration.
- `run_logs/layer02_compact_prompt_audit.json` records compact six-target source batches.
- Cache remains short-path and best-effort; cache I/O cannot invalidate a model answer.


## 4. Projected benchmark metrics and strict native span metrics


In [5]:
summary = analyze_run(
    run_dir=RUN_DIR,
    gold_jsonl=GOLD_JSONL,
    catalog_path=RELATION_CATALOG,
    aliases_path=RELATION_ALIASES,
)

display(pd.DataFrame(summary["layer_summary"]))

print("Projected benchmark relation evaluation; exact-span projection, null excluded")
display(pd.DataFrame([summary["strict_relation_evaluation"]]))

print("Strict native span relation evaluation; unmapped native predictions count as false positives")
display(pd.DataFrame([summary["native_span_relation_evaluation"]]))

print("Projected event-ID evaluation")
display(pd.DataFrame([summary["event_entity_evaluation"]]))

print("Strict native event-span evaluation")
display(pd.DataFrame([summary["native_span_event_evaluation"]]))

print("Relation-endpoint projected event evaluation")
display(pd.DataFrame([summary["relation_endpoint_evaluation"]]))

print("Relation candidate-pool coverage")
display(pd.DataFrame([summary["candidate_pool_evaluation"]]))

print("Per-relation projected metrics")
display(pd.DataFrame(summary["per_relation_metrics"]))

print("Direction/class confusion matrix")
display(pd.DataFrame(summary["relation_confusion_matrix"]))

print("Cumulative evaluation")
display(pd.DataFrame(summary["cumulative_evaluation"]))

print("First-failure counts")
print(summary["failure_counts"])


,layer,layer_name,linguistic_expressions,enriched_expressions,entity_candidates,relation_candidates,attribute_candidates,event_candidates,candidate_relation_assertions,candidate_triples,concept_candidates,ontology_relation_candidates,concept_hierarchy_links,relation_hierarchy_links,axiom_schema_candidates,general_axiom_candidates,completion_candidates,validation_issues,reasoning_inferred_triples
0,0,layer00_preprocessing,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,layer01_linguistic_expression_extraction,210,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,layer02_candidate_enrichment,210,129,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,layer03_candidate_typing_resolution,210,129,0,109,0,20,0,0,0,0,0,0,0,0,0,0,0
4,4,layer04_candidate_relation_extraction,210,129,0,109,0,20,109,0,0,0,0,0,0,0,0,0,0
5,5,layer05_candidate_triple_generation,210,129,0,109,0,20,109,109,0,0,0,0,0,0,0,0,0
6,6,layer06_concept_relation_induction,210,129,0,109,0,20,109,109,0,2,0,0,0,0,0,0,0
7,7,layer07_hierarchisation,210,129,0,109,0,20,109,109,0,2,0,2,0,0,0,0,0
8,8,layer08_axiom_schemata_extraction,210,129,0,109,0,20,109,109,0,2,0,2,6,0,0,0,0
9,9,layer09_general_axiom_extraction,210,129,0,109,0,20,109,109,0,2,0,2,6,8,0,0,0


Projected benchmark relation evaluation; exact-span projection, null excluded


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,39,20,6,33,14,0.153846,0.3,0.20339


Strict native span relation evaluation; unmapped native predictions count as false positives


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,109,20,6,103,14,0.055046,0.3,0.093023


...

Direction/class confusion matrix


,gold_relation,PRECONDITION,FALLING_ACTION,NONE,REVERSED_PRECONDITION,REVERSED_FALLING_ACTION,PAIR_MISSING,INVALID
0,PRECONDITION,1,3,0,0,0,3,0
1,FALLING_ACTION,1,5,4,0,0,3,0


Cumulative evaluation


,layer,layer_name,projected_relation_predicted,projected_relation_gold,projected_relation_true_positive,projected_relation_false_positive,projected_relation_false_negative,projected_relation_precision,projected_relation_recall,projected_relation_f1,...,projected_event_recall,projected_event_f1,native_span_event_predicted,native_span_event_gold,native_span_event_true_positive,native_span_event_false_positive,native_span_event_false_negative,native_span_event_precision,native_span_event_recall,native_span_event_f1
0,0,layer00_preprocessing,0,20,0,0,20,0.000000,0.0,0.00000,...,0.000000,0.000000,0,15,0,0,15,0.00,0.000000,0.000000
1,1,layer01_linguistic_expression_extraction,0,20,0,0,20,0.000000,0.0,0.00000,...,0.866667,0.928571,20,15,13,7,2,0.65,0.866667,0.742857
2,2,layer02_candidate_enrichment,0,20,0,0,20,0.000000,0.0,0.00000,...,0.866667,0.928571,20,15,13,7,2,0.65,0.866667,0.742857
3,3,layer03_candidate_typing_resolution,0,20,0,0,20,0.000000,0.0,0.00000,...,0.866667,0.928571,20,15,13,7,2,0.65,0.866667,0.742857
4,4,layer04_candidate_relation_extraction,39,20,6,33,14,0.153846,0.3,0.20339,...,0.866667,0.928571,20,15,13,7,2,0.65,0.866667,0.742857
5,5,layer05_candidate_triple_generation,39,20,6,33,14,0.153846,0.3,0.20339,...,0.866667,0.928571,20,15,13,7,2,0.65,0.866667,0.742857
6,6,layer06_concept_relation_induction,39,20,6,33,14,0.153846,0.3,0.20339,...,0.866667,0.928571,20,15,13,7,2,0.65,0.866667,0.742857
7,7,layer07_hierarchisation,39,20,6,33,14,0.153846,0.3,0.20339,...,0.866667,0.928571,20,15,13,7,2,0.65,0.866667,0.742857
8,8,layer08_axiom_schemata_extraction,39,20,6,33,14,0.153846,0.3,0.20339,...,0.866667,0.928571,20,15,13,7,2,0.65,0.866667,0.742857
9,9,layer09_general_axiom_extraction,39,20,6,33,14,0.153846,0.3,0.20339,...,0.866667,0.928571,20,15,13,7,2,0.65,0.866667,0.742857


First-failure counts
{'classified_none': 4, 'wrong_relation_class': 4, 'target_event_missing': 3, 'survived_to_layer05': 6, 'source_event_missing': 3}


## 5. Layer 1 validated event inventory and deterministic unordered relation-pair pool


In [6]:
layer1_rows = read_json(RUN_DIR / "run_logs/layer01_event_relation_instances.json")
layer1_df = pd.DataFrame(layer1_rows)
display(layer1_df)

accepted = layer1_df[layer1_df["status"] == "accepted"] if not layer1_df.empty else layer1_df
if not accepted.empty:
    print("Accepted event mentions:", int((accepted["label"] == "event_mention").sum()))
    print("Deterministic pair expressions:", int((accepted["label"] == "relation_instance").sum()))
    print("Rejected rows:", int((layer1_df["status"] == "rejected").sum()))


,phase,status,expr_id,text,label,pair_id,candidate_reasons
0,atomic_materialization,accepted,expr_00000,S1[2:4]::checks into,event_mention,NaN,NaN
1,atomic_materialization,accepted,expr_00001,S3[1:2]::skipping,event_mention,NaN,NaN
2,atomic_materialization,accepted,expr_00002,S3[4:5]::entering,event_mention,NaN,NaN
3,atomic_materialization,accepted,expr_00003,S3[11:12]::facing,event_mention,NaN,NaN
4,atomic_materialization,accepted,expr_00004,S3[15:16]::arrest,event_mention,NaN,NaN
...,...,...,...,...,...,...,...
205,atomic_pair_pool_materialization,accepted,expr_00205,S5[9:12]::rear - ended || potentially related ...,relation_instance,P00185,[exhaustive_unordered_pair]
206,atomic_pair_pool_materialization,accepted,expr_00206,S5[9:12]::rear - ended || potentially related ...,relation_instance,P00186,[exhaustive_unordered_pair]
207,atomic_pair_pool_materialization,accepted,expr_00207,S5[27:28]::lied || potentially related || S5[3...,relation_instance,P00187,[exhaustive_unordered_pair]
208,atomic_pair_pool_materialization,accepted,expr_00208,S5[27:28]::lied || potentially related || S5[3...,relation_instance,P00188,[exhaustive_unordered_pair]


Accepted event mentions: 20
Deterministic pair expressions: 190
Rejected rows: 0


### Layer 1A span validation/repair and Layer 1 deterministic pair audit


In [7]:
inventory_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_event_inventory.json"))
atomic_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_atomic_span_review.json"))
pair_pool = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_pair_pool.json"))
call_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_call_audit.json"))

print("Final atomic event inventory audit")
display(inventory_audit)
print("Atomic KEEP/DROP/REPLACE/SPLIT audit")
display(atomic_audit)

print("Deterministic unordered pair pool rebuilt after atomic refinement")
display(pair_pool)
if not pair_pool.empty:
    print("Initial pair candidates:", len(pair_pool))
    print("Pair strategy values:", sorted(pair_pool["strategy"].dropna().unique()))

print("Layer 1 sentence/review/atomic/pair construction audit")
display(call_audit)
if not call_audit.empty:
    display(call_audit.groupby(["phase", "status"], dropna=False).size().reset_index(name="calls"))


Final atomic event inventory audit


,phase,original_event_key,result_event_key,action,reasons,event_key,reason,status
0,deterministic_atomic_minimization,S1[2:4]::checks into,S1[2:4]::checks into,KEEP,[],NaN,NaN,NaN
1,deterministic_atomic_minimization,S3[1:2]::skipping,S3[1:2]::skipping,KEEP,[],NaN,NaN,NaN
2,deterministic_atomic_minimization,S3[4:5]::entering,S3[4:5]::entering,KEEP,[],NaN,NaN,NaN
3,deterministic_atomic_minimization,S3[11:12]::facing,S3[11:12]::facing,KEEP,[],NaN,NaN,NaN
4,deterministic_atomic_minimization,S3[15:16]::arrest,S3[15:16]::arrest,KEEP,[],NaN,NaN,NaN
5,deterministic_atomic_minimization,S3[17:18]::violating,S3[17:18]::violating,KEEP,[],NaN,NaN,NaN
6,deterministic_atomic_minimization,S3[24:26]::checked into,S3[24:26]::checked into,KEEP,[],NaN,NaN,NaN
7,deterministic_atomic_minimization,S3[31:32]::begin,S3[31:32]::begin,KEEP,[],NaN,NaN,NaN
8,deterministic_atomic_minimization,S3[33:36]::90 - day,S3[33:36]::90 - day,KEEP,[],NaN,NaN,NaN
9,deterministic_atomic_minimization,S3[36:39]::court - mandated,S3[36:39]::court - mandated,KEEP,[],NaN,NaN,NaN


Atomic KEEP/DROP/REPLACE/SPLIT audit


,phase,original_event_key,result_event_key,action,reasons,event_key,reason,status
0,deterministic_atomic_minimization,S1[2:4]::checks into,S1[2:4]::checks into,KEEP,[],NaN,NaN,NaN
1,deterministic_atomic_minimization,S3[1:2]::skipping,S3[1:2]::skipping,KEEP,[],NaN,NaN,NaN
2,deterministic_atomic_minimization,S3[4:5]::entering,S3[4:5]::entering,KEEP,[],NaN,NaN,NaN
3,deterministic_atomic_minimization,S3[11:12]::facing,S3[11:12]::facing,KEEP,[],NaN,NaN,NaN
4,deterministic_atomic_minimization,S3[15:16]::arrest,S3[15:16]::arrest,KEEP,[],NaN,NaN,NaN
5,deterministic_atomic_minimization,S3[17:18]::violating,S3[17:18]::violating,KEEP,[],NaN,NaN,NaN
6,deterministic_atomic_minimization,S3[24:26]::checked into,S3[24:26]::checked into,KEEP,[],NaN,NaN,NaN
7,deterministic_atomic_minimization,S3[31:32]::begin,S3[31:32]::begin,KEEP,[],NaN,NaN,NaN
8,deterministic_atomic_minimization,S3[33:36]::90 - day,S3[33:36]::90 - day,KEEP,[],NaN,NaN,NaN
9,deterministic_atomic_minimization,S3[36:39]::court - mandated,S3[36:39]::court - mandated,KEEP,[],NaN,NaN,NaN


Deterministic unordered pair pool rebuilt after atomic refinement


,event_a_id,event_b_id,event_a_key,event_b_key,event_a_sentence,event_b_sentence,candidate_reasons,strategy,phase,pair_id
0,E0000,E0001,S1[2:4]::checks into,S3[1:2]::skipping,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00000
1,E0000,E0002,S1[2:4]::checks into,S3[4:5]::entering,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00001
2,E0000,E0003,S1[2:4]::checks into,S3[11:12]::facing,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00002
3,E0000,E0004,S1[2:4]::checks into,S3[15:16]::arrest,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00003
4,E0000,E0005,S1[2:4]::checks into,S3[17:18]::violating,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00004
...,...,...,...,...,...,...,...,...,...,...
185,E0016,E0018,S5[9:12]::rear - ended,S5[31:32]::telling,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00185
186,E0016,E0019,S5[9:12]::rear - ended,S5[36:37]::driving,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00186
187,E0017,E0018,S5[27:28]::lied,S5[31:32]::telling,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00187
188,E0017,E0019,S5[27:28]::lied,S5[36:37]::driving,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00188


Initial pair candidates: 190
Pair strategy values: ['exhaustive']
Layer 1 sentence/review/atomic/pair construction audit


,phase,sentence_id,status,reason,proposals,new_validated_events,pass_index,inventory_size_after,strategy,event_count,pair_count,exhaustive_event_threshold,input_events,reviewed_events,dropped_events,replacement_or_added_events,final_events
0,sentence_inventory,0.0,skipped,url_sentence,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sentence_inventory,2.0,skipped,date_or_publication_metadata,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sentence_inventory,1.0,ok,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sentence_inventory,3.0,ok,NaN,11.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sentence_inventory,4.0,ok,NaN,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,sentence_inventory,5.0,ok,NaN,5.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,coverage_review,NaN,ok,NaN,2.0,0.0,1.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,coverage_review,NaN,ok,NaN,4.0,0.0,2.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,coverage_review,NaN,ok,NaN,2.0,0.0,3.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,deterministic_pair_pool,NaN,ok,NaN,NaN,NaN,NaN,NaN,exhaustive,20.0,190.0,25.0,NaN,NaN,NaN,NaN,NaN


,phase,status,calls
0,atomic_span_review,ok,1
1,coverage_review,ok,3
2,deterministic_pair_pool,ok,1
3,deterministic_pair_pool_after_atomic_review,ok,1
4,sentence_inventory,ok,4
5,sentence_inventory,skipped,2


## 6. Forward-only plot-calibrated relation classification


In [8]:
layer2 = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_relation_decisions.json"))
source_decisions = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_source_centric_decisions.json"))
source_calls = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_source_centric_call_audit.json"))
source_parse = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_source_parse_audit.json"))
prompt_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_compact_prompt_audit.json"))
conflict_audit = read_json(RUN_DIR / "run_logs/layer02_direction_conflicts.json")

plot_roles = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_plot_roles.json"))
forward_filter = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_forward_direction_filter.json"))
calibration = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_relation_calibration.json"))
print("Global plot roles")
display(plot_roles)
print("Forward orientation audit")
display(forward_filter)
print("Relation calibration audit")
display(calibration)
if not calibration.empty:
    changed = calibration[calibration["before"] != calibration["after"]]
    print("Calibrated relation labels:", len(changed))
    display(changed)
closure_pool = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_closure_pair_pool.json"))

print("Final one-per-unordered-pair Layer 2 decisions")
display(layer2)
if not layer2.empty:
    display(layer2.groupby(["status", "decision"], dropna=False).size().reset_index(name="pairs"))
    print("Positive controlled relations entering Layer 3:", int((layer2["status"] == "accepted").sum()))

print("All directed source-to-target decisions")
display(source_decisions)
if not source_decisions.empty:
    display(source_decisions.groupby("relation", dropna=False).size().reset_index(name="directed_decisions"))
    print("Directed target decisions:", len(source_decisions))
    print("Evidence-format repairs:", int(source_decisions.get("evidence_repaired", pd.Series(dtype=bool)).fillna(False).sum()))

print("Fixed-source call and focused recovery audit")
display(source_calls)
if not source_calls.empty:
    display(source_calls.groupby(["phase", "status"], dropna=False).agg(
        calls=("source_event_id", "count"),
        requested_targets=("requested_targets", "sum"),
        resolved_targets=("resolved_targets", "sum"),
        elapsed_seconds=("elapsed_seconds", "sum"),
    ).reset_index())

print("Prompt audit")
display(prompt_audit)
if not prompt_audit.empty:
    source_prompts = prompt_audit[prompt_audit["prompt_kind"] == "v17_forward_plot_calibrated_batch"]
    print("Primary/recovery source prompts:", len(source_prompts))
    if not source_prompts.empty:
        print("Mean targets per source prompt:", round(source_prompts["target_count"].mean(), 2))
        print("Mean user characters:", round(source_prompts["user_chars"].mean(), 1))
        print("Maximum user characters:", int(source_prompts["user_chars"].max()))

print("Direction conflicts (should be zero by construction)")
print("Conflict count:", conflict_audit.get("conflict_count", 0))
display(pd.DataFrame(conflict_audit.get("audit", [])))

print("Parser/normalization audit")
display(source_parse)
print("Candidate closure pairs (must be zero):", len(closure_pool))
assert len(closure_pool) == 0


Global plot roles


,event_id,event_key,role,branch_id,reason,confidence,meta_status
0,E0000,S1[2:4]::checks into,SETUP,B1,Introduces main character and setting,0.9,ok
1,E0001,S3[1:2]::skipping,SETUP,B1,"Describes skipping, establishing context",0.9,ok
2,E0002,S3[4:5]::entering,SETUP,B1,"Mentions entering facility, part of setup",0.9,ok
3,E0003,S3[11:12]::facing,SETUP,B1,"Facing arrest, part of motivation",0.9,ok
4,E0004,S3[15:16]::arrest,PIVOT,B1,Central threat of arrest drives action,0.9,ok
5,E0005,S3[17:18]::violating,SETUP,B1,Violation of probation explains arrest,0.9,ok
6,E0006,S3[24:26]::checked into,PIVOT,B1,Actual check‑in is key event,0.9,ok
7,E0007,S3[31:32]::begin,PIVOT,B1,Beginning of stay is focal point,0.9,ok
8,E0008,S3[33:36]::90 - day,AFTERMATH,B1,Details length of stay,0.8,ok
9,E0009,S3[36:39]::court - mandated,AFTERMATH,B1,Court‑mandated nature of stay,0.8,ok


Forward orientation audit


,pair_id,earlier_event,later_event,classified_direction,reverse_direction_skipped
0,P00000,S1[2:4]::checks into,S3[1:2]::skipping,S1[2:4]::checks into -> S3[1:2]::skipping,True
1,P00001,S1[2:4]::checks into,S3[4:5]::entering,S1[2:4]::checks into -> S3[4:5]::entering,True
2,P00002,S1[2:4]::checks into,S3[11:12]::facing,S1[2:4]::checks into -> S3[11:12]::facing,True
3,P00003,S1[2:4]::checks into,S3[15:16]::arrest,S1[2:4]::checks into -> S3[15:16]::arrest,True
4,P00004,S1[2:4]::checks into,S3[17:18]::violating,S1[2:4]::checks into -> S3[17:18]::violating,True
...,...,...,...,...,...
185,P00185,S5[9:12]::rear - ended,S5[31:32]::telling,S5[9:12]::rear - ended -> S5[31:32]::telling,True
186,P00186,S5[9:12]::rear - ended,S5[36:37]::driving,S5[9:12]::rear - ended -> S5[36:37]::driving,True
187,P00187,S5[27:28]::lied,S5[31:32]::telling,S5[27:28]::lied -> S5[31:32]::telling,True
188,P00188,S5[27:28]::lied,S5[36:37]::driving,S5[27:28]::lied -> S5[36:37]::driving,True


Relation calibration audit


...

Primary/recovery source prompts: 40
Mean targets per source prompt: 4.75
Mean user characters: 7607.4
Maximum user characters: 9254
Direction conflicts (should be zero by construction)
Conflict count: 0


""


Parser/normalization audit


,source_event_id,source_event_key,target_event_id,target_event_key,relation,evidence_sentence_ids,evidence_text,reason,confidence,phase,evidence_repaired,status
0,E0007,S3[31:32]::begin,E0008,S3[33:36]::90 - day,FALLING_ACTION,[3],...to begin a 90 - day court - mandated stay...,The 90‑day duration is a detail that follows t...,0.95,primary_forward_batch_00,False,parsed
1,E0007,S3[31:32]::begin,E0009,S3[36:39]::court - mandated,FALLING_ACTION,[3],...to begin a 90 - day court - mandated stay...,The court‑mandated nature explains the type of...,0.95,primary_forward_batch_00,False,parsed
2,E0007,S3[31:32]::begin,E0010,S3[39:40]::stay,FALLING_ACTION,[3],...to begin a 90 - day court - mandated stay...,The stay itself is the immediate consequence o...,0.95,primary_forward_batch_00,False,parsed
3,E0007,S3[31:32]::begin,E0011,S3[44:45]::conviction,FALLING_ACTION,[3],...in her reckless driving conviction...,The conviction provides context for the stay t...,0.90,primary_forward_batch_00,False,parsed
4,E0007,S3[31:32]::begin,E0012,S4[15:16]::violation,FALLING_ACTION,"[3, 4]",...after a probation violation in a prior drun...,The prior violation is background that explain...,0.85,primary_forward_batch_00,True,parsed
...,...,...,...,...,...,...,...,...,...,...,...,...
185,E0017,S5[27:28]::lied,E0019,S5[36:37]::driving,NONE,[],,The driving event is a background context of t...,0.85,primary_forward_batch_00,False,parsed
186,E0015,S5[6:7]::comes,E0016,S5[9:12]::rear - ended,NONE,[],,The source event (stay at Betty Ford) occurs a...,0.95,primary_forward_batch_00,False,parsed
187,E0015,S5[6:7]::comes,E0017,S5[27:28]::lied,NONE,[],,The source event is after the target event (li...,0.95,primary_forward_batch_00,False,parsed
188,E0015,S5[6:7]::comes,E0018,S5[31:32]::telling,NONE,[],,The source event precedes the target event (te...,0.95,primary_forward_batch_00,False,parsed


Candidate closure pairs (must be zero): 0


## 7. Exact gold-relation failure trace and confusion matrix


In [9]:
trace_df = pd.read_csv(RUN_DIR / "analysis/gold_relation_trace.csv")
confusion_df = pd.read_csv(RUN_DIR / "analysis/relation_confusion_matrix.csv")
display(trace_df)
display(
    trace_df.groupby("first_failure", dropna=False)
    .size()
    .reset_index(name="gold_relations")
    .sort_values("gold_relations", ascending=False)
)
display(confusion_df)


,source_key,relation_id,target_key,first_failure,pair_in_pool,decision_status,predicted_decision,predicted_source,predicted_relation,predicted_target
0,S1[2:4]::checks into,FALLING_ACTION,S3[15:16]::arrest,classified_none,True,filtered_none,NONE,NaN,NaN,NaN
1,S1[2:4]::checks into,FALLING_ACTION,S3[4:5]::entering,classified_none,True,filtered_none,NONE,NaN,NaN,NaN
2,S1[2:4]::checks into,PRECONDITION,S3[39:40]::stay,wrong_relation_class,True,accepted,A_FALLING_ACTION_B,S1[2:4]::checks into,FALLING_ACTION,S3[39:40]::stay
3,S1[2:4]::checks into,PRECONDITION,S5[2:3]::stay,target_event_missing,False,NaN,NaN,NaN,NaN,NaN
4,S3[15:16]::arrest,FALLING_ACTION,S3[17:18]::violating,classified_none,True,filtered_none,NONE,NaN,NaN,NaN
5,S3[15:16]::arrest,PRECONDITION,S3[24:26]::checked into,survived_to_layer05,True,accepted,A_PRECONDITION_B,S3[15:16]::arrest,PRECONDITION,S3[24:26]::checked into
6,S3[24:26]::checked into,PRECONDITION,S3[39:40]::stay,wrong_relation_class,True,accepted,A_FALLING_ACTION_B,S3[24:26]::checked into,FALLING_ACTION,S3[39:40]::stay
7,S3[24:26]::checked into,PRECONDITION,S5[2:3]::stay,target_event_missing,False,NaN,NaN,NaN,NaN,NaN
8,S3[39:40]::stay,FALLING_ACTION,S3[44:45]::conviction,survived_to_layer05,True,accepted,A_FALLING_ACTION_B,S3[39:40]::stay,FALLING_ACTION,S3[44:45]::conviction
9,S3[39:40]::stay,FALLING_ACTION,S5[27:28]::lied,survived_to_layer05,True,accepted,A_FALLING_ACTION_B,S3[39:40]::stay,FALLING_ACTION,S5[27:28]::lied


,first_failure,gold_relations
2,survived_to_layer05,6
0,classified_none,4
4,wrong_relation_class,4
1,source_event_missing,3
3,target_event_missing,3


,gold_relation,PRECONDITION,FALLING_ACTION,NONE,REVERSED_PRECONDITION,REVERSED_FALLING_ACTION,PAIR_MISSING,INVALID
0,PRECONDITION,1,3,0,0,0,3,0
1,FALLING_ACTION,1,5,4,0,0,3,0


## 8. Native event candidates, assertions and triples

In [10]:
states = {index: state for index, _, state in load_layer_states(RUN_DIR)}
layer3 = states.get(3)
layer4 = states.get(4)
layer5 = states.get(5)
layer6 = states.get(6)
layer11 = states.get(11)

if layer3:
    print("Layer 3 event candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "ontology_hints": c.ontology_hints,
    } for c in layer3.event_candidates or []]))

    print("Layer 3 relation candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "controlled_hints": [h for h in c.ontology_hints if str(h).lower().startswith("controlled_relation:")],
    } for c in layer3.relation_candidates or []]))
    assert all(c.mentions for c in layer3.relation_candidates or []), "Mention-free relation candidate detected."

if layer4:
    print("Layer 4 assertions")
    display(pd.DataFrame([{
        "source": x.source_candidate_label,
        "predicate": x.relation_label,
        "target": x.target_candidate_label,
        "confidence": x.confidence,
    } for x in layer4.candidate_relation_assertions or []]))

if layer5:
    print("Layer 5 triples")
    display(pd.DataFrame([{
        "subject": x.subject_label,
        "predicate": x.predicate_label,
        "object": x.object_label,
        "confidence": x.confidence,
    } for x in layer5.candidate_triples or []]))

print("Layer 6 ontology relation candidates:", len(layer6.ontology_relation_candidates or []) if layer6 else None)
print("Layer 11 completion candidates:", len(layer11.completion_candidates or []) if layer11 else None)

Layer 3 event candidates


,candidate_id,canonical_label,mentions,ontology_hints
0,cand_s_00000,S1[2:4]::checks into,[S1[2:4]::checks into],"[semantic_role:event_mention, candidate_family..."
1,cand_s_00001,S3[1:2]::skipping,[S3[1:2]::skipping],"[semantic_role:event_mention, candidate_family..."
2,cand_s_00002,S3[4:5]::entering,[S3[4:5]::entering],"[semantic_role:event_mention, candidate_family..."
3,cand_s_00003,S3[11:12]::facing,[S3[11:12]::facing],"[semantic_role:event_mention, candidate_family..."
4,cand_s_00004,S3[15:16]::arrest,[S3[15:16]::arrest],"[semantic_role:event_mention, candidate_family..."
5,cand_s_00005,S3[17:18]::violating,[S3[17:18]::violating],"[semantic_role:event_mention, candidate_family..."
6,cand_s_00006,S3[24:26]::checked into,[S3[24:26]::checked into],"[semantic_role:event_mention, candidate_family..."
7,cand_s_00007,S3[31:32]::begin,[S3[31:32]::begin],"[semantic_role:event_mention, candidate_family..."
8,cand_s_00008,S3[33:36]::90 - day,[S3[33:36]::90 - day],"[semantic_role:event_mention, candidate_family..."
9,cand_s_00009,S3[36:39]::court - mandated,[S3[36:39]::court - mandated],"[semantic_role:event_mention, candidate_family..."


Layer 3 relation candidates


,candidate_id,canonical_label,mentions,controlled_hints
0,cand_r_00000,FALLING_ACTION,[S1[2:4]::checks into || FALLING_ACTION || S3[...,[controlled_relation:FALLING_ACTION]
1,cand_r_00001,FALLING_ACTION,[S1[2:4]::checks into || FALLING_ACTION || S3[...,[controlled_relation:FALLING_ACTION]
2,cand_r_00002,FALLING_ACTION,[S1[2:4]::checks into || FALLING_ACTION || S3[...,[controlled_relation:FALLING_ACTION]
3,cand_r_00003,FALLING_ACTION,[S1[2:4]::checks into || FALLING_ACTION || S3[...,[controlled_relation:FALLING_ACTION]
4,cand_r_00004,FALLING_ACTION,[S1[2:4]::checks into || FALLING_ACTION || S3[...,[controlled_relation:FALLING_ACTION]
...,...,...,...,...
104,cand_r_00104,FALLING_ACTION,[S4[20:21]::driving || FALLING_ACTION || S5[31...,[controlled_relation:FALLING_ACTION]
105,cand_r_00105,FALLING_ACTION,[S5[9:12]::rear - ended || FALLING_ACTION || S...,[controlled_relation:FALLING_ACTION]
106,cand_r_00106,FALLING_ACTION,[S5[9:12]::rear - ended || FALLING_ACTION || S...,[controlled_relation:FALLING_ACTION]
107,cand_r_00107,FALLING_ACTION,[S5[27:28]::lied || FALLING_ACTION || S5[31:32...,[controlled_relation:FALLING_ACTION]


Layer 4 assertions


,source,predicate,target,confidence
0,S1[2:4]::checks into,FALLING_ACTION,S3[31:32]::begin,1.0
1,S1[2:4]::checks into,FALLING_ACTION,S3[33:36]::90 - day,1.0
2,S1[2:4]::checks into,FALLING_ACTION,S3[36:39]::court - mandated,1.0
3,S1[2:4]::checks into,FALLING_ACTION,S3[39:40]::stay,1.0
4,S1[2:4]::checks into,FALLING_ACTION,S3[44:45]::conviction,1.0
...,...,...,...,...
104,S4[20:21]::driving,FALLING_ACTION,S5[31:32]::telling,1.0
105,S5[9:12]::rear - ended,FALLING_ACTION,S5[27:28]::lied,1.0
106,S5[9:12]::rear - ended,FALLING_ACTION,S5[31:32]::telling,1.0
107,S5[27:28]::lied,FALLING_ACTION,S5[31:32]::telling,1.0


Layer 5 triples


,subject,predicate,object,confidence
0,S1[2:4]::checks into,FALLING_ACTION,S3[31:32]::begin,1.0
1,S1[2:4]::checks into,FALLING_ACTION,S3[33:36]::90 - day,1.0
2,S1[2:4]::checks into,FALLING_ACTION,S3[36:39]::court - mandated,1.0
3,S1[2:4]::checks into,FALLING_ACTION,S3[39:40]::stay,1.0
4,S1[2:4]::checks into,FALLING_ACTION,S3[44:45]::conviction,1.0
...,...,...,...,...
104,S4[20:21]::driving,FALLING_ACTION,S5[31:32]::telling,1.0
105,S5[9:12]::rear - ended,FALLING_ACTION,S5[27:28]::lied,1.0
106,S5[9:12]::rear - ended,FALLING_ACTION,S5[31:32]::telling,1.0
107,S5[27:28]::lied,FALLING_ACTION,S5[31:32]::telling,1.0


Layer 6 ontology relation candidates: 2
Layer 11 completion candidates: 0


## 9. Exact-span-only event projection audit


In [11]:
projection = pd.read_csv(RUN_DIR / "analysis/event_projection_audit.csv")
display(projection)

# Exact-span demonstration using source structure. Gold is consulted only here,
# after the pipeline has completed. Trigger-only fallbacks are disabled.
first_gold_key = next(iter(gold_event_index(gold)["keys_by_id"].values()))[0]
print(first_gold_key)
print(project_event_label(first_gold_key, gold))


,event_id,method,label,candidate_event_ids
0,EVENT_34cccabacad4dea93d3e762114dc05cd,exact_sentence_token_span,S1[2:4]::checks into,NaN
1,NaN,unmapped_or_ambiguous_exact_span,S3[1:2]::skipping,[]
2,EVENT_b79df11f8737f700691af4f8a7132190,exact_sentence_token_span,S3[4:5]::entering,NaN
3,NaN,unmapped_or_ambiguous_exact_span,S3[11:12]::facing,[]
4,EVENT_c332a6b60af0495f34856f112dc68632,exact_sentence_token_span,S3[15:16]::arrest,NaN
5,EVENT_9c07ef70f303f23dd82b32fb28a61ecb,exact_sentence_token_span,S3[17:18]::violating,NaN
6,EVENT_d7aec1e4ff20c3264f9fb7686355eb8a,exact_sentence_token_span,S3[24:26]::checked into,NaN
7,NaN,unmapped_or_ambiguous_exact_span,S3[31:32]::begin,[]
8,NaN,unmapped_or_ambiguous_exact_span,S3[33:36]::90 - day,[]
9,NaN,unmapped_or_ambiguous_exact_span,S3[36:39]::court - mandated,[]


S4[15:16]::violation
{'event_id': 'EVENT_434faad0c9ec71baa6a91df3c385299b', 'method': 'exact_sentence_token_span', 'label': 'S4[15:16]::violation'}


## 10. Speed, concurrency and errors

In [12]:
def optional_jsonl(path: Path) -> pd.DataFrame:
    return pd.DataFrame(read_jsonl(path)) if path.is_file() else pd.DataFrame()

calls = optional_jsonl(RUN_DIR / "run_logs/llm_calls.jsonl")
errors = optional_jsonl(RUN_DIR / "run_logs/llm_errors.jsonl")
parse_errors = optional_jsonl(RUN_DIR / "run_logs/llm_parse_errors.jsonl")
retrieval = optional_jsonl(RUN_DIR / "run_logs/ontology_retrieval.jsonl")

if not calls.empty:
    display(calls)
    display(calls.groupby("layer_tag").agg(
        calls=("call_index", "count"),
        total_recorded_seconds=("elapsed_seconds", "sum"),
        maximum_call_seconds=("elapsed_seconds", "max"),
        mean_system_chars=("system_chars", "mean"),
        mean_user_chars=("user_chars", "mean"),
        mean_response_chars=("response_chars", "mean"),
    ).reset_index())

print("Backend/API errors:", len(errors))
if not errors.empty: display(errors)
print("JSON parse errors:", len(parse_errors))
if not parse_errors.empty: display(parse_errors)
if not retrieval.empty:
    display(retrieval.groupby("layer_name").size().reset_index(name="retrieval_calls"))

manifest = read_json(RUN_DIR / "run_manifest.json")
print("Wall-clock pipeline seconds:", manifest.get("elapsed_seconds"))

,call_index,layer_tag,model,temperature,message_count,system_chars,user_chars,max_tokens,request_timeout,started_at,status,elapsed_seconds,response_chars,response_path,json_parse_ok,parsed_type,parse_error
0,1,layer01_atomic_event_inventory_v1_7,openai/gpt-oss-20b,0.0,2,1675,2281,8192,180,2026-08-18 14:08:28,ok,1.132,206,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
1,2,layer01_atomic_event_inventory_v1_7,openai/gpt-oss-20b,0.0,2,1675,2841,8192,180,2026-08-18 14:08:28,ok,1.427,1594,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
2,3,layer01_atomic_event_inventory_v1_7,openai/gpt-oss-20b,0.0,2,1675,2500,8192,180,2026-08-18 14:08:28,ok,3.420,468,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
3,4,layer01_atomic_event_inventory_v1_7,openai/gpt-oss-20b,0.0,2,1675,2651,8192,180,2026-08-18 14:08:28,ok,3.806,739,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
4,5,layer01_atomic_event_inventory_v1_7,openai/gpt-oss-20b,0.0,2,986,5842,8192,180,2026-08-18 14:08:32,ok,2.167,351,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
5,6,layer01_atomic_event_inventory_v1_7,openai/gpt-oss-20b,0.0,2,1016,5842,8192,180,2026-08-18 14:08:34,ok,20.230,642,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
6,7,layer01_atomic_event_inventory_v1_7,openai/gpt-oss-20b,0.0,2,1027,5842,8192,180,2026-08-18 14:08:54,ok,7.175,351,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
7,8,layer01_atomic_event_inventory_v1_7,openai/gpt-oss-20b,0.0,2,1098,5065,8192,180,2026-08-18 14:09:02,ok,11.015,1077,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
8,9,layer02_forward_plot_relations_v1_7,openai/gpt-oss-20b,0.0,2,848,5704,4096,180,2026-08-18 14:09:13,ok,23.041,2256,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
9,12,layer02_forward_plot_relations_v1_7,openai/gpt-oss-20b,0.0,2,2761,8291,4096,180,2026-08-18 14:09:37,ok,1.944,2912,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None


,layer_tag,calls,total_recorded_seconds,maximum_call_seconds,mean_system_chars,mean_user_chars,mean_response_chars
0,layer01_atomic_event_inventory_v1_7,8,50.372,20.230,1353.375000,4108.0,678.500000
1,layer02_forward_plot_relations_v1_7,41,796.234,185.833,2714.341463,7561.0,1891.707317


Backend/API errors: 0
JSON parse errors: 0
Wall-clock pipeline seconds: 291.5259761810303


## Success checklist before the five-document batch

1. Atomic event extraction and deterministic pair coverage remain frozen.
2. Every unordered pair is oriented only from the earlier mention to the later mention; reverse causal reinterpretation is impossible.
3. One global plot-role call assigns SETUP/PIVOT/AFTERMATH/BACKGROUND soft roles.
4. Layer 2 classifies at most six forward targets per source prompt.
5. Narrative PLOT_LINK evidence is slightly relaxed: explicit physical causality is not required, but chronology/shared topic alone still fails.
6. PRECONDITION bias may be calibrated toward FALLING_ACTION only when downstream/plot-role evidence agrees.
7. `NONE` and malformed outputs never enter Layer 3.
8. Reverse-direction conflict adjudication is disabled because reverse directions are not queried.
9. Gold remains unavailable until post-Layer-12 evaluation.
10. Compare v1.7 against v1.6.3 on this same development document before freezing for smoke-5.
